# SatQuery AI - retrain: VQA adapter (rs_vqa_v1)

Qwen2.5-VL-3B QLoRA on RSVQA-LR-2k + designed refusals.

**Settings first:** Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified Kaggle account).

**Run with "Save Version -> Save & Run All (Commit)"** so it keeps running with the browser closed.

This notebook runs in two stages in one session:
1. **Smoke pass** (~20-40 min): the real pipeline with tiny limits. If anything is broken - versions, GPU, data, NaN losses - it stops here instead of hours later.
2. **Real run** (8-10 h, estimate). Training stops at 10.5 h after the session started, so Kaggle's 12 h limit never loses the work.

**Output** (Output tab -> Download): `retrained/` holds the weights, `retrain_manifest_vqa.json` and logs. `retrained_smoke/` is only the smoke evidence - do not install it.

If the VQA smoke pass reports a non-finite loss, change `FP32 = False` to `True` in the next cell and run again.

Docs: `docs/kaggle-retrain.md` in https://github.com/hs-zz27/sih2


In [ ]:
import time, subprocess
SESSION_START = time.time()
FP32 = False  # VQA only: set True if the smoke pass reports a non-finite loss
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
!rm -rf /kaggle/tmp/sih2 && git clone --depth 1 https://github.com/hs-zz27/sih2.git /kaggle/tmp/sih2
!cd /kaggle/tmp/sih2 && git log --oneline -1

In [ ]:
import json, pathlib
def check(path, need_complete):
    m = pathlib.Path(path)
    if not m.exists():
        raise SystemExit(f"NO MANIFEST at {m} - the driver did not start; read the cell above")
    d = json.loads(m.read_text())
    print("STATUS:", d["status"])
    print("training check:", d.get("training_check"))
    env = d.get("environment") or {}
    print("GPU:", env.get("gpu"), "| versions:", env.get("versions"))
    for k in ("metrics.json", "metrics_after_budget.json"):
        if k in d: print(k, json.dumps(d[k])[:400])
    ok = d["status"].startswith(need_complete)
    if not ok:
        raise SystemExit("STOPPING: " + d["status"])


## 1. Smoke pass

In [ ]:
fp32 = '--fp32-compute' if FP32 and 'vqa' == 'vqa' else ''
!cd /kaggle/tmp/sih2 && python training/kaggle/retrain.py --model vqa --smoke {fp32} --work /kaggle/tmp/satquery --out /kaggle/working/retrained_smoke
check('/kaggle/working/retrained_smoke/retrain_manifest_vqa.json', 'smoke complete')

## 2. Real run

In [ ]:
!cd /kaggle/tmp/sih2 && python training/kaggle/retrain.py --model vqa {fp32} --skip-install --session-start {SESSION_START} --work /kaggle/tmp/satquery --out /kaggle/working/retrained
check('/kaggle/working/retrained/retrain_manifest_vqa.json', ('complete', 'stopped by budget'))
!du -sh /kaggle/working/retrained/checkpoints

## 3. Score it on the official RSVQA-LR split

Without this the retrained adapter has no quotable accuracy - the Phase 5
number (0.8947) belongs to the lost weights, not to this one.

10,004 questions over 100 images, so this is roughly an hour on a T4. It runs
only if the training stage packaged an adapter, and the same session budget
applies. If the session is nearly spent, skip it: open a fresh notebook later
with only this cell, pointing `--adapter` at the downloaded adapter.

In [ ]:
import pathlib
adapter = pathlib.Path('/kaggle/working/retrained/checkpoints/v2/track_b_vqa/adapter_final')
if not adapter.exists():
    print('no adapter packaged - skipping the official evaluation')
else:
    !cd /kaggle/tmp/sih2 && python training/kaggle/retrain.py --model eval_vqa \
        --adapter {adapter} --skip-install --session-start {SESSION_START} \
        --work /kaggle/tmp/satquery --out /kaggle/working/retrained
    check('/kaggle/working/retrained/retrain_manifest_eval_vqa.json',
          ('complete', 'stopped by budget'))